# EDA — Application table

Reference: PLAN_v2.md §5 — lightweight scaffolding only. The competition has been EDA'd to death by public kernels; we focus on the few checks that actually drive modeling decisions.

**Checks performed:**
1. Target balance (~8.07% positive — confirm).
2. Missingness pattern across all columns.
3. Cardinality of categorical columns.
4. Distribution sanity for `AMT_INCOME_TOTAL`, `AMT_CREDIT`, `DAYS_*`.
5. The famous `DAYS_EMPLOYED == 365243` sentinel.
6. Correlation of top numerics with `TARGET`.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import polars as pl
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src import config
from src.data import read_raw

sns.set_style('whitegrid')
pd.set_option('display.max_columns', 120)

In [ ]:
train = read_raw('application_train')
test = read_raw('application_test')
print(f'train shape: {train.shape}')
print(f'test  shape: {test.shape}')

## 1. Target balance

In [ ]:
tgt = train['TARGET'].value_counts().sort('TARGET')
pos_rate = train['TARGET'].mean()
print(tgt)
print(f'Positive rate: {pos_rate:.4f}  →  scale_pos_weight ≈ {(1-pos_rate)/pos_rate:.2f}')

## 2. Missingness pattern

In [ ]:
null_counts = train.null_count().transpose(include_header=True, header_name='column', column_names=['n_null']).sort('n_null', descending=True)
null_counts = null_counts.with_columns((pl.col('n_null') / train.height).alias('null_rate'))
null_counts.head(30)

In [ ]:
high_null = null_counts.filter(pl.col('null_rate') > 0.5)
print(f'{high_null.height} columns are >50% null. PLAN §2.9: drop only if >99%.')

## 3. Cardinality of categoricals

In [ ]:
cat_cols = [c for c, dt in train.schema.items() if dt == pl.Utf8]
card = pl.DataFrame({
    'col': cat_cols,
    'n_unique': [train[c].n_unique() for c in cat_cols],
}).sort('n_unique', descending=True)
print(f'{len(cat_cols)} categorical columns')
card

## 4. DAYS_EMPLOYED sentinel

The `365243` value (≈1000 years) is the documented sentinel for unemployed/missing. Confirm the count and the bimodal distribution.

In [ ]:
n_sentinel = (train['DAYS_EMPLOYED'] == 365243).sum()
print(f'DAYS_EMPLOYED == 365243 count: {n_sentinel:,}  ({n_sentinel/train.height:.2%})')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(train['DAYS_EMPLOYED'].drop_nulls(), bins=80)
axes[0].set_title('DAYS_EMPLOYED — raw (note the spike at 365243)')
valid = train.filter(pl.col('DAYS_EMPLOYED') != 365243)['DAYS_EMPLOYED'].drop_nulls()
axes[1].hist(valid, bins=80)
axes[1].set_title('DAYS_EMPLOYED — sentinel removed')
plt.tight_layout()
plt.show()

## 5. AMT_INCOME_TOTAL outliers

In [ ]:
income = train['AMT_INCOME_TOTAL']
print(f'min={income.min():,.0f}, p99={income.quantile(0.99):,.0f}, max={income.max():,.0f}')
print(f'Records with income > 1M: {(income > 1_000_000).sum()}')

## 6. Correlation of top numerics with TARGET

Sanity check: `EXT_SOURCE_*` should dominate the absolute-correlation ranking.

In [ ]:
numeric_cols = [c for c, dt in train.schema.items() if dt.is_numeric() and c not in ('SK_ID_CURR', 'TARGET')]
tdf = train.select(numeric_cols + ['TARGET']).to_pandas()
corrs = tdf.corr()['TARGET'].drop('TARGET').abs().sort_values(ascending=False)
corrs.head(25)

**Expected**: `EXT_SOURCE_3`, `EXT_SOURCE_2`, `EXT_SOURCE_1`, `DAYS_BIRTH`, `DAYS_EMPLOYED` at the top. If `EXT_SOURCE_*` are NOT in the top 5, something is wrong with the load.